In [ ]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24" "transformers>=4.40.0" "torch>=2.3.0" "accelerate>=0.29.0"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

In [ ]:
# Step 2: Load FLARE-FPB test set and normalize labels
from datasets import load_dataset, Dataset

LABELS = ["negative", "neutral", "positive"]

ds_raw = load_dataset("TheFinAI/flare-fpb", split="test")
print("Loaded flare-fpb test:", len(ds_raw), "columns:", ds_raw.column_names)

_alias = {"pos": "positive", "neg": "negative", "neu": "neutral",
          "bullish": "positive", "bearish": "negative"}

def _norm_label(v):
    if v is None: 
        return None
    if isinstance(v, (int, float)) or (isinstance(v, str) and v.isdigit()):
        i = int(v)
        return LABELS[i] if 0 <= i < len(LABELS) else None
    s = str(v).strip().lower()
    s = _alias.get(s, s)
    return s if s in LABELS else None

def _map_row(x):
    text = x.get("text") or x.get("sentence") or x.get("content") or x.get("input") or ""
    lab = _norm_label(x.get("label", x.get("labels", x.get("answer"))))
    return {"text": text, "choices": LABELS, "answer": lab}

ds = Dataset.from_list([{**r, **_map_row(r)} for r in ds_raw])
bad = [i for i, r in enumerate(ds) if r["answer"] not in LABELS]
print("Samples with unusable label:", len(bad))
assert len(bad) == 0, "Found unparseable labels; please check the field mapping."

In [ ]:
# Step 3: Install dependencies and record experiment metadata
%pip install -q "pandas>=2.2.2" "tqdm>=4.66.4" "scikit-learn>=1.4.0"

import os, json, time, platform, torch
from importlib.metadata import version, PackageNotFoundError

# Gemma-3-4B-IT配置
MODEL_NAME = "google/gemma-3-4b-it"
MODEL_SHORT = "gemma-3-4b-it"

# 检查GPU可用性
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# 适配文件命名
run_tag = f"flare_fpb_{MODEL_SHORT.replace('-', '_').replace('/', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

# 更新元数据
meta = {
    "dataset": "TheFinAI/flare-fpb",
    "split": "test",
    "labels": list(LABELS),
    "model": MODEL_NAME,
    "model_short": MODEL_SHORT,
    "provider": "Google",
    "device": device,
    "transformers_version": ver("transformers"),
    "torch_version": ver("torch"),
    "accelerate_version": ver("accelerate"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "note": "Gemma-3-4B-IT model evaluation with Hugging Face Transformers"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL_NAME)
print("Device:", device)

In [ ]:
# Step 4: Load model and tokenizer, then run inference
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import gc

# 清理GPU缓存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print(f"Loading model: {MODEL_NAME}")
print("This may take a few minutes...")

# 加载tokenizer和模型
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ["HF_TOKEN"])

# 加载模型配置
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token=os.environ["HF_TOKEN"]
)

# 将模型移动到设备
if not torch.cuda.is_available():
    model = model.to(device)

model.eval()
print("Model loaded successfully!")

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_chat_template(text: str, choices=("negative", "neutral", "positive")):
    """为Gemma-3创建对话格式"""
    # Gemma-3使用特殊的对话格式
    prompt = f"""<start_of_turn>user
Task: classify the sentence into exactly one of these labels: {', '.join(choices)}.

Sentence: {text}

Return ONLY a JSON object on a single line, exactly in this form:
{{"label":"negative|neutral|positive"}}
No code fences, no extra text, no explanation.<end_of_turn>
<start_of_turn>model
"""
    return prompt

def ask_gemma_once(text, choices=("negative", "neutral", "positive"), max_new_tokens=100):
    """使用Gemma-3-4B-IT进行推理"""
    prompt = _make_chat_template(text, choices)
    
    try:
        # 编码输入
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
        
        # 移动到设备
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 生成输出
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,  # 低温度以获得确定性的输出
                do_sample=True,
                top_p=0.95,
                top_k=50,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        # 解码输出
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # 提取模型回复部分
        if "<start_of_turn>model" in generated_text:
            response = generated_text.split("<start_of_turn>model")[-1].strip()
        else:
            response = generated_text[len(prompt):].strip()
        
        # 清理响应
        response = _strip_code_fences(response)
        
        # 解析JSON
        try:
            # 尝试直接解析
            obj = json.loads(response)
        except json.JSONDecodeError:
            # 尝试提取JSON对象
            match = re.search(r'\{.*?\}', response, re.DOTALL)
            if match:
                obj = json.loads(match.group())
            else:
                raise RuntimeError(f"Could not parse JSON from response: {response[:200]}")
        
        lab = obj.get("label")
        
        if lab not in choices:
            raise RuntimeError(f"Invalid label {lab!r}; raw json: {obj}")
        
        return lab
        
    except Exception as e:
        raise RuntimeError(f"Gemma inference error: {str(e)}")

def ask_gemma(text, choices=("negative", "neutral", "positive")):
    """带重试机制的Gemma调用"""
    for max_tokens in (100, 200, 300):
        delay = 2.0
        for attempt in range(5):
            try:
                return ask_gemma_once(text, choices, max_new_tokens=max_tokens)
            except RuntimeError as e:
                msg = str(e).lower()
                
                # 处理内存不足错误
                if "memory" in msg or "cuda" in msg or "out of memory" in msg:
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        gc.collect()
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                # 处理JSON解析错误
                if "json" in msg or "parse" in msg:
                    time.sleep(delay)
                    delay = min(delay * 1.5, 30)
                    continue
                raise
            except Exception as e:
                time.sleep(delay)
                delay = min(delay * 2, 60)
                if attempt < 4:
                    continue
                raise
    raise RuntimeError("Exhausted retries for this sample.")

run_tag = f"flare_fpb_{MODEL_SHORT.replace('-', '_').replace('/', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"

rows_done = []
done_idx = set()
if os.path.exists(pred_path):
    old = pd.read_csv(pred_path)
    if "row_idx" in old.columns:
        rows_done = old.to_dict("records")
        done_idx = set(old["row_idx"].tolist())
        print(f"[resume] loaded {len(done_idx)} completed rows.")

err_rows = []
buf = []
save_every = 10  # 减少保存频率，因为本地推理更快

total = len(ds)
print(f"Starting Gemma-3-4B-IT model evaluation on {total} samples...")

# 定期清理GPU缓存
def cleanup_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

for i in tqdm(range(total)):
    if i in done_idx:
        continue
    
    # 每50个样本清理一次GPU
    if i % 50 == 0 and i > 0:
        cleanup_gpu()
    
    x = ds[i]
    text = x["text"]
    gold = x["answer"]

    try:
        pred = ask_gemma(text, LABELS)
        raw = json.dumps({"label": pred})
    except Exception as e:
        pred = "UNKNOWN"
        raw = f"ERROR: {type(e).__name__}: {e}"
        err_rows.append({"row_idx": i, "id": x.get("id", i), "error": raw, "text": text})

    buf.append({
        "row_idx": i,
        "id": x.get("id", i),
        "text": text,
        "pred_raw": raw,
        "pred": pred,
        "label": gold
    })

    if len(buf) % save_every == 0:
        out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
        out.to_csv(pred_path, index=False)
        if err_rows:
            pd.DataFrame(err_rows).to_csv(err_path, index=False)
        print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

# 最终保存
cleanup_gpu()
out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
out.to_csv(pred_path, index=False)
if err_rows:
    pd.DataFrame(err_rows).to_csv(err_path, index=False)
print(f"[done] Gemma-3-4B-IT evaluation completed -> {pred_path}")
if os.path.exists(err_path):
    err_count = len(pd.read_csv(err_path)) if os.path.getsize(err_path) > 0 else 0
    print(f"[errors] {err_count} errors logged -> {err_path}")

In [ ]:
# Step 5: Compute evaluation metrics
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import json
import time

# 加载预测结果
df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")
ok = df[df["pred"] != "UNKNOWN"].copy()

print(f"Gemma-3-4B-IT Model Evaluation Results:")
print(f"Total samples: {len(df)}")
print(f"Successful predictions: {len(ok)}")
print(f"Failed predictions: {len(df) - len(ok)}")

if len(ok) > 0:
    # 计算评估指标
    f1_macro = f1_score(ok["label"], ok["pred"], labels=LABELS, average="macro", zero_division=0)
    f1_micro = f1_score(ok["label"], ok["pred"], labels=LABELS, average="micro", zero_division=0)
    f1_weighted = f1_score(ok["label"], ok["pred"], labels=LABELS, average="weighted", zero_division=0)
    accuracy = accuracy_score(ok["label"], ok["pred"])
    
    print("\n" + "="*50)
    print("EVALUATION RESULTS - Gemma-3-4B-IT")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1-Macro:  {f1_macro:.4f}")
    print(f"F1-Micro:  {f1_micro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    
    # 详细分类报告
    print("\nDetailed Classification Report:")
    print(classification_report(ok["label"], ok["pred"], labels=LABELS, zero_division=0))
    
    # 混淆矩阵
    print("Confusion Matrix:")
    cm = confusion_matrix(ok["label"], ok["pred"], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    print(cm_df)
    
    # 保存评估结果
    eval_results = {
        "model": MODEL_NAME,
        "model_short": MODEL_SHORT,
        "dataset": "TheFinAI/flare-fpb",
        "split": "test",
        "device": device,
        "total_samples": len(df),
        "successful_predictions": len(ok),
        "failed_predictions": len(df) - len(ok),
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro),
        "f1_weighted": float(f1_weighted),
        "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "confusion_matrix": cm.tolist(),
        "labels": LABELS
    }
    
    eval_path = f"{save_dir}/{run_tag}_evaluation_results.json"
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
    print(f"\nEvaluation results saved -> {eval_path}")
    
else:
    print("No successful predictions to evaluate!")